# 분리수거 오류 수정 v2 — 데이터 검토·평가·실험 준비
기존 모델/원본 파일은 보존합니다. 이미지를 확인한 train 두 장에서만 중복된 뚜껑 박스를 제거하는 패치를 제공합니다. 화면 전체 paper 탐지를 숨기기 위해 큰 박스를 삭제하거나 confidence를 임의로 높이지 않습니다.

**이번 작업에서 재학습하지 않았으며 이 노트북의 학습 셀도 계획만 출력합니다.** 기존 best.pt의 성능 개선을 주장하지 않습니다. 나머지 라벨 후보는 사람이 검토해야 합니다. 원래 데이터셋·추가 데이터·v1 패치는 실제 제공 파일을 사용해야 합니다.

## 1. 경로 설정
현재 클라우드 환경의 Python 인터프리터는 `/workspace/onboarding/venv/bin/python`입니다. 다른 환경에서는 `requirements.txt`를 참고해 필요한 패키지를 설치하고 아래 경로를 조정하세요. Colab에서는 INPUTS를 `/content` 등 실제 업로드 위치로 변경합니다. 새 RUN 폴더를 생성하며 이전 실행 결과를 덮어쓰지 않습니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, sys, json, csv
CODE_DIR = Path('/workspace/my-codex-test/수정안_v2')
INPUTS = Path('/workspace/attachments')
WEIGHTS = Path('/workspace/recycling_assets/label_fix_v1/weights/best.pt')
RUN = Path('/workspace/recycling_v2_runs') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
SCRIPT = CODE_DIR / 'recycling_fix.py'
assert SCRIPT.is_file()
def find_one(pattern):
    found = list(INPUTS.rglob(pattern))
    if len(found) != 1:
        raise FileNotFoundError(f'실제 파일 하나를 지정해야 합니다: {pattern}; 발견 {len(found)}개')
    return found[0]
STARTER = find_one('recycling_yolo_starter*.zip')
SUPPLEMENT = find_one('recycling_supplement_v1.zip')
PATCH_V1 = find_one('recycling_label_patch_v1*.json')
PATCH_V2 = CODE_DIR / 'reviewed_label_fix_v2.json'
def run(*args):
    subprocess.run([sys.executable, str(SCRIPT), *map(str,args)], check=True)
print('이번 실행 폴더:', RUN)

## 2. label_fix_v1 데이터 구성 재현 → 검토된 두 장만 수정
이전 patch JSON이 포함한 27개 수정을 재현한 baseline 사본과, 이번에 확인한 2개 수정만 추가한 사본을 구분합니다. 두 사본의 val/test 이미지는 바이트 단위로 같습니다. 기존 원본을 수정하지 않습니다.

In [ ]:
BASE = RUN / 'baseline_data'
REVIEWED = RUN / 'reviewed_data'
run('prepare', '--starter', STARTER, '--supplement', SUPPLEMENT,
    '--patch', PATCH_V1, '--out', BASE)
run('apply-reviewed', '--data', BASE/'data.yaml', '--patch', PATCH_V2, '--out', REVIEWED)
run('audit', '--data', BASE/'data.yaml', '--out', RUN/'baseline_audit')
run('audit', '--data', REVIEWED/'data.yaml', '--out', RUN/'reviewed_audit')

## 3. 남은 라벨 후보 시각 검토
포함 관계는 오류의 증거가 아니라 검토 후보입니다. 자동 삭제하지 않습니다. 아래는 후보 하나를 원본 박스와 함께 보여줍니다. class 숫자 및 좌표는 실제 파일에서 읽습니다. 검토 전후 평가 라벨을 섞어 성능이 개선됐다고 주장하지 마세요.

In [ ]:
import sys
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from recycling_fix import parse_labels, xyxy
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
with (RUN/'reviewed_audit/label_review.csv').open(encoding='utf-8-sig') as f:
    candidates = list(csv.DictReader(f))
print('train 후보 사진 수:', len({r['image'] for r in candidates if r['split']=='train'}))
# train 검토를 먼저 진행합니다. test를 모델 조정용으로 반복 사용하지 않습니다.
train_candidates = [r for r in candidates if r['split']=='train']
INDEX = 0
if train_candidates:
    row = train_candidates[INDEX]
    with Image.open(row['image']) as raw:
        image = raw.convert('RGB')
    labels = parse_labels(Path(row['label']).read_text(encoding='utf-8'))
    fig, ax = plt.subplots(figsize=(7,7))
    ax.imshow(image)
    for number, label in enumerate(labels,1):
        x1,y1,x2,y2 = xyxy(label)
        ax.add_patch(Rectangle((x1*image.width,y1*image.height),
                              (x2-x1)*image.width,(y2-y1)*image.height,
                              fill=False,edgecolor='red',linewidth=2))
        ax.text(x1*image.width,y1*image.height,f'box {number}, class {label[0]}',color='red')
    ax.set_title(Path(row['image']).name); plt.show()

## 4. 기존 모델 평가
현재 best.pt로 val을 평가합니다. train 라벨을 바꾸는 것만으로 기존 best.pt는 바뀌지 않습니다. 이 결과는 새 모델의 성능이 아닙니다. 원본 데이터의 캐시도 바꾸지 않도록 평가 데이터 사본을 임시 폴더에 사용합니다.

In [ ]:
run('evaluate', '--weights', WEIGHTS, '--data', REVIEWED/'data.yaml',
    '--split', 'val', '--out', RUN/'old_model_val')

## 5. 실제 원본 사진 추론
박스·글자가 그려진 annotated JPG를 원본 대신 넣지 마세요. 클래스 이름은 모델에서 읽고, EXIF 회전 보정과 RGB 변환을 적용합니다. 결과가 0개면 그대로 0개를 기록합니다. 큰 paper 박스를 강제로 지우는 규칙은 없습니다.

In [ ]:
PHOTOS = []  # 실제 원본 사진 경로를 순서대로 지정하세요.
if not PHOTOS:
    print('미실행: 실제 원본 사진을 지정해야 합니다.')
else:
    run('predict', '--weights', WEIGHTS, '--out', RUN/'real_photos', '--images', *PHOTOS)

## 6. 배경 오탐 평가 — 독립 사진만 사용
NEGATIVE_EVAL_DIR에는 사용자가 물체 0개임을 확인한 실제 사진만 넣습니다. 학습에 사용한 배경은 독립 평가에서 차단합니다. 빈 배경에는 mAP나 recall을 계산하지 않고, 오탐이 발생한 사진 비율과 사진당 오탐 박스 수를 측정합니다. 배경 평가 사진은 이후 학습 자료로 옮기지 마세요.

In [ ]:
NEGATIVE_EVAL_DIR = None  # 예: Path('/content/independent_empty_backgrounds')
CONFIRMED_EMPTY = False
if NEGATIVE_EVAL_DIR is not None and CONFIRMED_EMPTY:
    run('evaluate', '--weights', WEIGHTS, '--data', REVIEWED/'data.yaml',
        '--split', 'val', '--negative-eval-dir', NEGATIVE_EVAL_DIR,
        '--confirm-negatives', '--out', RUN/'val_and_background')
else:
    print('미실행: 독립 배경 사진 및 객체 0개 확인이 필요합니다.')

## 7. 다음 실험 계획 — 재학습 실행 없음
baseline과 reviewed_data를 같은 사전학습 모델·seed·epochs로 각각 학습해야 라벨 수정 효과를 비교할 수 있습니다. Negative Sample의 효과는 별도 조건으로 비교하세요. 아래에는 --execute가 없으므로 모델 초기화나 학습을 실행하지 않습니다. 남은 라벨 검토가 필요합니다.

In [ ]:
run('train-plan', '--data', REVIEWED/'data.yaml',
    '--pretrained', Path('/workspace/recycling_assets/yolo11n.pt'),
    '--out', RUN/'future_training')